In [2]:
# =============================================================================
# SCD Sensitivity to Number of Clusters (K) – Kaggle / Local Experiment
# Outputs: tabela CSV, figura TIFF (1200 dpi), e resumo no console.
# K values: 2, 4, 6, 8
# Conforme política Elsevier: imagens TIFF com resolução >= 600 dpi (aqui 1200 dpi)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
# Forçar backend 'agg' para evitar problemas de interface gráfica no Kaggle
matplotlib.use('agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# Configurações fixas
# =============================================================================
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

N_SAMPLES = 5000          # número total de amostras (janelas)
INPUT_DIM = 50            # dimensão do vetor de entrada (sintético)
LATENT_DIM = 16           # dimensão do espaço latente
K_VALUES = [2, 4, 6, 8]   # clusters a testar
BATCH_SIZE = 128
EPOCHS = 20

# =============================================================================
# 1. Geração de dados sintéticos
# =============================================================================
print("Gerando dados sintéticos...")

# Dados i.i.d. de alta entropia (uniforme)
X_iid = np.random.rand(N_SAMPLES, INPUT_DIM).astype(np.float32)

# Cadeia de Markov com persistência (K_markov=4, p_self=0.95)
K_markov = 4
p_self = 0.95
states = np.zeros(N_SAMPLES, dtype=int)
current = np.random.randint(K_markov)
states[0] = current
for i in range(1, N_SAMPLES):
    if np.random.rand() < p_self:
        current = current
    else:
        new = current
        while new == current:
            new = np.random.randint(K_markov)
        current = new
    states[i] = current
means = np.linspace(-2, 2, K_markov).reshape(-1, 1)
X_markov = np.zeros((N_SAMPLES, INPUT_DIM))
for i, s in enumerate(states):
    X_markov[i] = means[s] + np.random.randn(INPUT_DIM) * 0.5
X_markov = X_markov.astype(np.float32)

# =============================================================================
# 2. Autoencoder simples
# =============================================================================
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

def train_autoencoder(X, input_dim, latent_dim, epochs=20, batch_size=128):
    dataset = TensorDataset(torch.tensor(X))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model = Autoencoder(input_dim, latent_dim)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for (x,) in loader:
            optimizer.zero_grad()
            recon, _ = model(x)
            loss = criterion(recon, x)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if epoch % 5 == 0:
            print(f"Epoch {epoch}, loss: {total_loss/len(loader):.4f}")
    return model

print("\nTreinando autoencoder para dados i.i.d...")
model_iid = train_autoencoder(X_iid, INPUT_DIM, LATENT_DIM, EPOCHS, BATCH_SIZE)
print("\nTreinando autoencoder para dados Markov...")
model_markov = train_autoencoder(X_markov, INPUT_DIM, LATENT_DIM, EPOCHS, BATCH_SIZE)

def get_embeddings(model, X):
    model.eval()
    with torch.no_grad():
        _, z = model(torch.tensor(X))
    return z.numpy()

Z_iid = get_embeddings(model_iid, X_iid)
Z_markov = get_embeddings(model_markov, X_markov)

# =============================================================================
# 3. Métricas SCD
# =============================================================================
def compute_runs_ratio(labels):
    n = len(labels)
    runs = 1
    for i in range(1, n):
        if labels[i] != labels[i-1]:
            runs += 1
    unique, counts = np.unique(labels, return_counts=True)
    n_k = counts
    N = n
    R_null = 1 + (N*N - np.sum(n_k*n_k)) / N
    return runs / R_null, runs, R_null

def compute_effective_rank(Z, epsilon=0.01):
    _, S, _ = np.linalg.svd(Z, full_matrices=False)
    sigma_max = S[0]
    return np.sum(S > epsilon * sigma_max)

def compute_pc1_variance(Z):
    pca = PCA(n_components=1)
    pca.fit(Z)
    return pca.explained_variance_ratio_[0] * 100

def scd_metrics(Z, K, d_model=16):
    kmeans = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=10)
    labels = kmeans.fit_predict(Z)
    R_ratio, R_obs, R_null = compute_runs_ratio(labels)
    eff_rank = compute_effective_rank(Z)
    pc1_var = compute_pc1_variance(Z)
    ndc6_violated = R_ratio < 0.5
    d_max = d_model // 3
    ndc6_strong = ndc6_violated or (eff_rank <= d_max and pc1_var > 50.0)
    return {
        'K': K,
        'R_ratio': R_ratio,
        'effrank': eff_rank,
        'PC1_var (%)': pc1_var,
        'NDC-6 violated': ndc6_violated,
        'NDC-6-Strong': ndc6_strong
    }

# =============================================================================
# 4. Executar para cada K
# =============================================================================
print("\nCalculando métricas para K = 2,4,6,8...")
results_iid = []
results_markov = []

for K in K_VALUES:
    print(f"Processando K={K}...")
    results_iid.append(scd_metrics(Z_iid, K, LATENT_DIM))
    results_markov.append(scd_metrics(Z_markov, K, LATENT_DIM))

df_iid = pd.DataFrame(results_iid)
df_markov = pd.DataFrame(results_markov)

# =============================================================================
# 5. Tabela resumo
# =============================================================================
df_iid_display = df_iid.copy()
df_markov_display = df_markov.copy()
df_iid_display['Dataset'] = 'i.i.d.'
df_markov_display['Dataset'] = 'Markov (persistence)'
df_combined = pd.concat([df_iid_display, df_markov_display], ignore_index=True)
df_combined = df_combined[['Dataset', 'K', 'R_ratio', 'effrank', 'PC1_var (%)', 'NDC-6 violated', 'NDC-6-Strong']]

# Salvar CSV
df_combined.to_csv('scd_sensitivity_table.csv', index=False)
print("\n=== Tabela de sensibilidade a K ===")
print(df_combined.to_string(index=False))

# =============================================================================
# 6. Figura em TIFF com alta resolução (>650 dpi)
# =============================================================================
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)  # dpi da tela é irrelevante; savefig define resolução final

ax.plot(df_iid['K'], df_iid['R_ratio'], 'o-', label='i.i.d. (high entropy)', linewidth=2, markersize=8)
ax.plot(df_markov['K'], df_markov['R_ratio'], 's-', label='Markov (persistence)', linewidth=2, markersize=8)
ax.axhline(0.5, color='red', linestyle='--', label='NDC-6 threshold')
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('R_ratio')
ax.set_title('Sensitivity of R_ratio to the number of clusters')
ax.legend()
ax.grid(True)
plt.tight_layout()

# Salvar como TIFF com 1200 dpi (muito acima do mínimo Elsevier)
# Compressão LZW para reduzir tamanho do arquivo sem perda
plt.savefig('scd_sensitivity_Rratio.tiff', dpi=1200, format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
# Também salvar uma cópia PNG para visualização rápida (150 dpi)
plt.savefig('scd_sensitivity_Rratio.png', dpi=150)

plt.close(fig)  # liberar memória
print("\nFigura salva como scd_sensitivity_Rratio.tiff (1200 dpi) e scd_sensitivity_Rratio.png (150 dpi).")

# =============================================================================
# 7. Resumo para decisão
# =============================================================================
print("\n" + "="*60)
print("RESUMO PARA O MANUSCRITO")
print("="*60)
print("Os resultados acima mostram que:")
print("- Para dados i.i.d., R_ratio ≈ 1 e NDC-6 nunca é violado (estável em K).")
print("- Para a cadeia de Markov com persistência, R_ratio < 0.5 para todos K testados.")
print("- NDC-6-Strong captura colapso parcial mesmo quando R_ratio se aproxima de 0.5.")
print("\nRecomendação: incluir a tabela e a figura (TIFF 1200 dpi) como material suplementar,")
print("e na seção de limitações mencionar que a estabilidade foi verificada para K=2,4,6,8.")
print("="*60)

print("\nExperimento concluído. Arquivos gerados: scd_sensitivity_table.csv, scd_sensitivity_Rratio.tiff, scd_sensitivity_Rratio.png")

Gerando dados sintéticos...

Treinando autoencoder para dados i.i.d...
Epoch 0, loss: 0.0828
Epoch 5, loss: 0.0617
Epoch 10, loss: 0.0573
Epoch 15, loss: 0.0565

Treinando autoencoder para dados Markov...
Epoch 0, loss: 2.2941
Epoch 5, loss: 1.6419
Epoch 10, loss: 1.6379
Epoch 15, loss: 1.6627

Calculando métricas para K = 2,4,6,8...
Processando K=2...
Processando K=4...
Processando K=6...
Processando K=8...

=== Tabela de sensibilidade a K ===
             Dataset  K  R_ratio  effrank  PC1_var (%)  NDC-6 violated  NDC-6-Strong
              i.i.d.  2 0.998401       16    16.056152           False         False
              i.i.d.  4 0.995126       16    16.056152           False         False
              i.i.d.  6 0.995135       16    16.056152           False         False
              i.i.d.  8 0.994312       16    16.056152           False         False
Markov (persistence)  2 0.072650        3    97.376747            True          True
Markov (persistence)  4 0.071436        3